In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/geoproject'

In [ ]:
# R-PROTOCOL -- written before any data is harvested; never edited afterwards.
import os, json, datetime
ROOT = '/content/drive/MyDrive/geoproject'
OUT = f'{ROOT}/osmfinal'
os.makedirs(OUT, exist_ok=True)
P = f'{OUT}/protocol.json'
assert not os.path.exists(P), 'protocol already written -- it is never overwritten'
protocol = {
  'written_utc': datetime.datetime.utcnow().isoformat() + 'Z',
  'source': 'OpenStreetMap description tags (ODbL), labelled by the element primary tag mapped to the 227-tag lexicon',
  'pipeline': ['en_core_web_trf extraction', 'guided extended FP-Growth', 'Bayesian elastic net selection', 'blended ranking'],
  'primary_metric': 'Hits@3', 'secondary_metrics': ['nDCG@20', 'Hits@1'],
  'tie_rule': 'worst-case: rank = #{score >= gold score}',
  'split': 'by OSM element and by name, 80/10/10, stratified by tag; Pakistan slice kept in each split; test id list hashed into split_stats.json at split time',
  'hyperparameters_chosen_on': 'dev only (lambda, min_support, EN C/l1_ratio, backbone); 106 and PK sets never used for selection',
  'adoption_rule': 'keep a change only if the paired-bootstrap 95% CI of dev Hits@3 difference excludes zero in its favour; otherwise keep the simpler/existing option',
  'prespecified_comparisons_holm': ['freq_en vs unselected P(tag|pattern)', 'bayes_en vs unselected P(tag|pattern)',
                                    'OSM-trained vs Wikivoyage-trained tables', 'w2v vs numberbatch',
                                    'final pipeline vs base-paper reproduction'],
  'test_policy': 'scored exactly once, guarded by a flag file',
  'learning_curve_fractions': [0.10, 0.25, 0.50, 1.00],
}
json.dump(protocol, open(P, 'w'), indent=1)
print('wrote', P, protocol['written_utc'])

In [ ]:
# R0a -- endpoint probe: how many OSM elements carry a description, before harvesting anything
import requests
EP = 'https://qlever.cs.uni-freiburg.de/api/osm-planet'
PFX = 'PREFIX osmkey: <https://www.openstreetmap.org/wiki/Key:>\n'
def q(sparql, timeout=600):
    r = requests.post(EP, data={'query': PFX + sparql}, headers={'Accept': 'application/sparql-results+json'}, timeout=timeout)
    r.raise_for_status(); return r.json()['results']['bindings']
res = q('SELECT (COUNT(?o) AS ?n) WHERE { ?o osmkey:description ?d }')
print('elements with description:', res[0]['n']['value'])

In [ ]:
# R0b -- schema probe: every predicate on one described cafe, so the harvest query is built from what exists
rows = q('''SELECT ?o WHERE { ?o osmkey:description ?d . ?o osmkey:amenity "cafe" } LIMIT 1''')
o = rows[0]['o']['value']; print('element:', o)
for b in q(f'SELECT ?p ?v WHERE {{ <{o}> ?p ?v }}'):
    print(f"{b['p']['value'][:80]:80s} {b['v']['value'][:70]}")

In [ ]:
# R0c -- Pakistan boundary relation (looked up, not assumed) + description counts per candidate key
pk = q('''SELECT ?r ?n WHERE { ?r osmkey:admin_level "2" ; osmkey:boundary "administrative" ;
          osmkey:ISO3166-1 "PK" . OPTIONAL { ?r osmkey:name:en ?n } }''')
print('Pakistan relation:', pk)
KEYS = ['amenity','shop','tourism','leisure','historic','natural','man_made','railway',
        'building','place','landuse','aeroway','waterway','highway','craft','office']
for k in KEYS:
    n = q(f'SELECT (COUNT(?o) AS ?n) WHERE {{ ?o osmkey:description ?d ; osmkey:{k} ?v }}')[0]['n']['value']
    print(f'{k:10s} {int(n):>9,}')

In [ ]:
# R0d -- Pakistan boundary via its wikidata tag (Q843), confirmed by name
pk = q('''SELECT ?r ?n ?lvl WHERE { ?r osmkey:wikidata "Q843" ; osmkey:admin_level ?lvl .
          OPTIONAL { ?r osmkey:name ?n } }''')
for b in pk: print(b['r']['value'], b.get('n', {}).get('value'), b['lvl']['value'])
PK_REL = [b['r']['value'] for b in pk if b['lvl']['value'] == '2']
assert len(PK_REL) == 1, 'expected exactly one admin_level 2 relation for Pakistan'
PK_REL = PK_REL[0]
n = q(f'SELECT (COUNT(?o) AS ?n) WHERE {{ ?o osmkey:description ?d ; <http://www.opengis.net/rdf#sfIntersects> <{PK_REL}> }}')
print('described elements inside Pakistan:', n[0]['n']['value'])

In [ ]:
# R0e -- harvest. Only the element id, its tag key/value, description, name and religion are
# fetched: no user, uid, phone, email or any other personal field ever enters the dataset.
import pandas as pd, time
os.makedirs(f'{OUT}/raw', exist_ok=True)
lit = lambda b, k: b[k]['value'] if k in b else None
frames = []
for k in KEYS:
    t0 = time.time()
    rows = q(f'''SELECT ?o ?v ?d ?name ?rel WHERE {{ ?o osmkey:description ?d ; osmkey:{k} ?v .
                 OPTIONAL {{ ?o osmkey:name ?name }} OPTIONAL {{ ?o osmkey:religion ?rel }} }}''', timeout=1800)
    df = pd.DataFrame({'osm_id': [b['o']['value'].replace('https://www.openstreetmap.org/', '') for b in rows],
                       'key': k, 'value': [lit(b, 'v') for b in rows], 'description': [lit(b, 'd') for b in rows],
                       'name': [lit(b, 'name') for b in rows], 'religion': [lit(b, 'rel') for b in rows]})
    frames.append(df); print(f'{k:10s} {len(df):>9,} rows  {time.time()-t0:5.0f}s')
raw = pd.concat(frames, ignore_index=True)
pk_ids = {b['o']['value'].replace('https://www.openstreetmap.org/', '') for b in
          q(f'SELECT ?o WHERE {{ ?o osmkey:description ?d ; <http://www.opengis.net/rdf#sfIntersects> <{PK_REL}> }}')}
raw['pakistan'] = raw['osm_id'].isin(pk_ids)
raw.to_parquet(f'{OUT}/raw/osm_descriptions.parquet')
print(f'\ntotal {len(raw):,} rows | {raw.osm_id.nunique():,} elements | Pakistan rows {raw.pakistan.sum():,}')

In [ ]:
# R1a -- direct mapping coverage, then the largest unmapped key=value pairs (the evidence for any extra map)
LEX = json.load(open(f'{ROOT}/eval/evalset_106.json'))['lexicon']
norm = lambda s: s.lower().replace('_', '-').replace(' ', '-') if isinstance(s, str) else None
LEXSET = set(LEX)
raw['vnorm'] = raw['value'].map(norm)
raw['tag'] = raw['vnorm'].where(raw['vnorm'].isin(LEXSET))
hit = raw['tag'].notna()
print(f'direct match: {hit.sum():,} rows ({hit.mean():.1%}), {raw.loc[hit, "tag"].nunique()} of {len(LEX)} lexicon tags')
um = raw.loc[~hit].groupby(['key', 'value']).size().sort_values(ascending=False)
print('\ntop 60 unmapped key=value:')
print('\n'.join(f'{n:>8,}  {k}={v}' for (k, v), n in um.head(60).items()))
print('\nlexicon tags with no direct match:', sorted(LEXSET - set(raw.loc[hit, 'tag'])))

In [ ]:
# R1b -- explicit map for spelling variants and the religion composite only, then one row per element
EXTRA = {('highway', 'cycleway'): 'cycle-way', ('highway', 'footway'): 'foot-way',
         ('natural', 'wetland'): 'wet-land', ('natural', 'wood'): 'woods', ('natural', 'sinkhole'): 'sink-hole',
         ('amenity', 'arts_centre'): 'arts-center', ('building', 'greenhouse'): 'green-house',
         ('building', 'houseboat'): 'house-boat', ('man_made', 'wastewater_plant'): 'waste-water',
         ('shop', 'books'): 'bookshop', ('place', 'province'): 'province', ('place', 'ocean'): 'ocean',
         ('place', 'continent'): 'continent'}
RELIGION = {'muslim': 'mosque', 'christian': 'church', 'buddhist': 'temple', 'hindu': 'temple',
            'jain': 'temple', 'sikh': 'temple', 'shinto': 'temple', 'taoist': 'temple'}
EXTRA = {kv: t for kv, t in EXTRA.items() if t in LEXSET}
RELIGION = {r: t for r, t in RELIGION.items() if t in LEXSET}
kv = list(zip(raw['key'], raw['value']))
raw['tag'] = raw['tag'].fillna(pd.Series([EXTRA.get(x) for x in kv], index=raw.index))
pow_ = (raw['key'] == 'amenity') & (raw['value'] == 'place_of_worship')
raw.loc[pow_, 'tag'] = raw.loc[pow_, 'religion'].map(RELIGION)

# one row per element: an element whose keys map to more than one distinct tag is ambiguous and dropped
m = raw[raw['tag'].notna()]
ntag = m.groupby('osm_id')['tag'].nunique()
keep = ntag[ntag == 1].index
el = m[m['osm_id'].isin(keep)].drop_duplicates('osm_id')[['osm_id', 'tag', 'key', 'value', 'description', 'name', 'pakistan']]
print(f'mapped rows {len(m):,} | elements {m.osm_id.nunique():,} | ambiguous dropped {(ntag > 1).sum():,} | kept {len(el):,}')
print(f'tags covered {el.tag.nunique()} of {len(LEX)} | Pakistan elements {el.pakistan.sum():,}')
vc = el['tag'].value_counts()
print('\nlargest:', vc.head(15).to_dict()); print('smallest:', vc.tail(15).to_dict())

In [ ]:
import py3langid, py3langid.langid as L, inspect
print(py3langid.__version__ if hasattr(py3langid, '__version__') else '')
print([n for n in dir(L.LanguageIdentifier) if not n.startswith('_')])
print([n for n in dir(L) if 'MODEL' in n or 'load' in n.lower()])
print(inspect.signature(L.LanguageIdentifier.__init__))

In [ ]:
print(inspect.signature(L.LanguageIdentifier.from_model_file)); print(L.MODEL_FILE)
t = L.LanguageIdentifier.from_model_file(L.MODEL_FILE, norm_probs=True)
print(t.classify('A quiet old stone chapel beside the river.'), t.classify('Ein kleines Café am Fluss.'))

In [ ]:
# R1d -- eyeball masked rows, and check overlap with the 106 and PK evaluation sets (P2.6 rule)
from collections import Counter
C = pd.read_parquet(f'{OUT}/osm_corpus.parquet')
for r in C.sample(12, random_state=1).itertuples():
    print(f'{r.tag:>14s} | {r.text[:110]}')
EV106 = json.load(open(f'{ROOT}/eval/evalset_106_extracted.json')); EV106 = EV106['rows'] if isinstance(EV106, dict) else EV106
EVPK = json.load(open(f'{ROOT}/eval/pk_evalset_extracted.json')); EVPK = EVPK['rows'] if isinstance(EVPK, dict) else EVPK
tok = lambda s: set(re.findall(r'[a-z]{3,}', s.lower()))
corpus_toks = [tok(t) for t in C['text']]
inv = {}
for i, s in enumerate(corpus_toks):
    for w in s: inv.setdefault(w, []).append(i)
def near(sentence):
    q_ = tok(sentence); cand = Counter(i for w in q_ for i in inv.get(w, []))
    return [i for i, c in cand.items() if c / len(q_ | corpus_toks[i]) >= 0.6]
for name, rows in (('106', EV106), ('PK', EVPK)):
    hits = sum(1 for r in rows if near(r['sentence']))
    print(f'{name}: {hits} of {len(rows)} evaluation sentences have a near-duplicate (Jaccard >= .6) in the OSM corpus')

In [ ]:
import importlib.util, platform, torch
print('python', platform.python_version(), '| cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
for m in ('spacy', 'spacy_transformers', 'en_core_web_trf', 'cupy'):
    s = importlib.util.find_spec(m); print(f'{m:20s}', 'installed' if s else 'MISSING')
if importlib.util.find_spec('spacy'):
    import spacy; print('spacy', spacy.__version__)

In [ ]:
# R2-setup -- install the transformer pipeline, then restart the kernel so spaCy registers its factories
import subprocess, sys
r = subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_trf'], capture_output=True, text=True)
print(r.stdout[-800:], r.stderr[-800:])
import os; os.kill(os.getpid(), 9)

In [ ]:
# R2 -- extraction with the thesis extractor, unchanged: en_core_web_trf + PlaceInterpreter.pipe via extract_rows
import sys, json, time, pandas as pd, spacy
ROOT = '/content/drive/MyDrive/geoproject'; OUT = f'{ROOT}/osmfinal'
sys.path.insert(0, ROOT)
import pipeline_ranker as pr, adjective_extractor as ae
print('gpu:', spacy.prefer_gpu())
C = pd.read_parquet(f'{OUT}/osm_corpus.parquet')
I = ae.PlaceInterpreter(model='en_core_web_trf')
t0 = time.time()
rows = pr.extract_rows(C['text'].tolist(), I, batch_size=256)
print(f'extracted {len(rows):,} rows in {time.time()-t0:.0f}s')
C['context_nouns'] = [json.dumps(r['context_nouns']) for r in rows]
C['adjectives']    = [json.dumps(r['adjectives']) for r in rows]
C['transaction']   = [json.dumps(r['transaction']) for r in rows]
C.to_parquet(f'{OUT}/osm_extracted.parquet')
has_adj = sum(1 for r in rows if r['adjectives'])
print(f'rows with >=1 adjective: {has_adj:,} ({has_adj/len(rows):.1%}) | mean nouns {sum(len(r["context_nouns"]) for r in rows)/len(rows):.2f}')

In [ ]:
# R3-source -- locate the cells that built p1_feat_meta.npz (patvocab / n_adj), print them verbatim
import glob
nbs = sorted(glob.glob('/content/drive/MyDrive/**/*.ipynb', recursive=False) + glob.glob('/content/drive/MyDrive/Colab Notebooks/*.ipynb'))
print(len(nbs), 'notebooks:', [n.split('/')[-1] for n in nbs])
for nb in nbs:
    try: cells = json.load(open(nb))['cells']
    except Exception as e: print('skip', nb, e); continue
    for i, c in enumerate(cells):
        src = ''.join(c.get('source', []))
        if c.get('cell_type') == 'code' and ('patvocab' in src or 'MIN_SUPPORT' in src):
            print(f'==== {nb.split("/")[-1]} cell {i} ====\n{src[:3000]}\n')

In [ ]:
# R3 -- guided FP-Growth on OSM train: P1.1 support rule + P1.2 cleanup, unchanged. Features written in
# the same npz layout Tables() already reads, so the ranker is reused without modification.
import numpy as np
from collections import Counter
from scipy import sparse
D = pd.read_parquet(f'{OUT}/osm_extracted.parquet')
for c in ('adjectives', 'transaction', 'context_nouns'): D[c] = D[c].map(json.loads)
D['adjs'] = D['adjectives'].map(lambda L: sorted({a.lower() for a in L if str(a).isalpha()}))   # P1.1 adjs_of rule
D['pit']  = D['transaction'].map(lambda L: [it for it in L if it.startswith('amod:')])          # P1.1 amod_items
tr = D[D.split == 'train']
N = len(tr); MIN_SUP = 0.0005; MIN_CNT = MIN_SUP * N
c1 = Counter(a for L in tr.adjs for a in L); c2 = Counter(p for L in tr.pit for p in L)
def _ok(p):                                                                                        # P1.2 cleanup
    body = p.split(':', 1)[1] if ':' in p else p
    return all(len(t) > 1 and t.isalnum() for t in body.split('_'))
FREQ_ADJ  = sorted(a for a, c in c1.items() if c >= MIN_CNT and _ok(a))
FREQ_PAIR = sorted(p for p, c in c2.items() if c >= MIN_CNT and _ok(p))
PV = FREQ_ADJ + FREQ_PAIR; PIX = {p: j for j, p in enumerate(PV)}; ADJ_SET, PAIR_SET = set(FREQ_ADJ), set(FREQ_PAIR)
CLASSES = sorted(D.tag.unique()); CIX = {c: k for k, c in enumerate(CLASSES)}
print(f'MIN_SUPPORT {MIN_SUP} (>= {MIN_CNT:.0f} of {N:,} train rows): {len(FREQ_ADJ)} adjective 1-itemsets + {len(FREQ_PAIR)} adj-noun 2-itemsets = {len(PV)} | {len(CLASSES)} classes')

def featurize(df):
    ri, ci = [], []
    for r, row in enumerate(df.itertuples(index=False)):
        seen = {PIX[a] for a in row.adjs if a in ADJ_SET} | {PIX[p] for p in row.pit if p in PAIR_SET}
        ri += [r] * len(seen); ci += list(seen)
    return sparse.csr_matrix((np.ones(len(ri), np.float32), (ri, ci)), shape=(len(df), len(PV)))
X, Y = {}, {}
for s in ('train', 'dev', 'test'):
    part = D[D.split == s]; X[s] = featurize(part); Y[s] = part.tag.map(CIX).values
    sparse.save_npz(f'{OUT}/X_{s}.npz', X[s])
    print(f'{s:5s} {X[s].shape}  rows with >=1 pattern {(X[s].getnnz(1) > 0).mean():.1%}')
np.savez(f'{OUT}/osm_feat_meta.npz', patvocab=np.array(PV), classes=np.array(CLASSES), n_adj=len(FREQ_ADJ),
         ytr=Y['train'], ydv=Y['dev'], yte=Y['test'], min_support=MIN_SUP)
print('top adjectives:', sorted(FREQ_ADJ, key=lambda a: -c1[a])[:20])
print('top pairs     :', sorted(FREQ_PAIR, key=lambda p: -c2[p])[:12])

In [ ]:
# R4a -- selection on the same 657 features: unselected association tables, and the frequentist EN (P1.3
# solver and l1_ratio, C over a grid; dev chooses C in R5). Every table is stored patterns x classes.
import time
from sklearn.linear_model import LogisticRegression
P, K = len(PV), len(CLASSES)
Xt, yt = X['train'], Y['train']
Cnt = np.zeros((P, K)); r_, c_ = Xt.nonzero()
np.add.at(Cnt, (c_, yt[r_]), 1)
Ntot = Cnt.sum(); rt = Cnt.sum(1, keepdims=True); ct = Cnt.sum(0, keepdims=True)
CONF = np.divide(Cnt, rt, out=np.zeros_like(Cnt), where=rt > 0)
with np.errstate(divide='ignore', invalid='ignore'):
    lift = (Cnt / Ntot) / ((rt / Ntot) * (ct / Ntot))
LOGLIFT = np.where(np.isfinite(lift) & (lift > 1), np.log(lift), 0.0)
tables = {'conf': CONF.astype(np.float32), 'loglift': LOGLIFT.astype(np.float32)}
for Cval in (0.1, 0.5, 2.0):
    t0 = time.time()
    m = LogisticRegression(penalty='elasticnet', solver='saga', l1_ratio=0.5, C=Cval, max_iter=200, tol=1e-3, n_jobs=-1).fit(Xt, yt)
    coef = np.zeros((P, K), np.float32); coef[:, m.classes_] = m.coef_.T
    tables[f'fen_C{Cval}'] = coef
    print(f'freq EN C={Cval}: {time.time()-t0:.0f}s, {int(np.max(m.n_iter_))} iters, nonzero {(m.coef_ != 0).mean():.1%}')
np.savez(f'{OUT}/tables_r4a.npz', **tables)
inv = dict(enumerate(PV))
for name in ('conf', 'loglift', 'fen_C0.5'):
    M = tables[name]; top = sorted(((M[i, j], inv[i], CLASSES[j]) for i, j in zip(*np.nonzero(M > 0))), reverse=True)[:8]
    print(f'top {name}:', [(a, c, round(float(v), 2)) for v, a, c in top])

In [ ]:
# R4b -- Bayesian elastic net, P1.7b scale-mixture model unchanged, NUTS 2 chains x 500+500 on the A100,
# fitted on every train row that carries >= 1 pattern (rows with none inform only the intercept).
import subprocess, sys, time
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'numpyro', 'multipledispatch'], check=True)
import jax, jax.numpy as jnp, numpyro, numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS
from numpyro.diagnostics import gelman_rubin, effective_sample_size
assert jax.default_backend() == 'gpu', jax.default_backend()
has = np.where(X['train'].getnnz(1) > 0)[0]
Xb = jnp.asarray(X['train'][has].toarray(), jnp.float32); yb = jnp.asarray(Y['train'][has]); Nb = len(has)
LAM2 = 0.5
def model_sm(X_, y_=None):
    b = numpyro.sample('b', dist.HalfNormal(1.0))
    tau2 = numpyro.sample('tau2', dist.Exponential(1.0 / (2.0 * b ** 2)).expand([P, K]).to_event(2))
    z = numpyro.sample('z', dist.Normal(0.0, 1.0).expand([P, K]).to_event(2))
    beta = numpyro.deterministic('beta', z * jnp.sqrt(tau2))
    numpyro.factor('ridge', -LAM2 * jnp.sum(beta ** 2))
    alpha = numpyro.sample('alpha', dist.Normal(0.0, 2.0).expand([K]).to_event(1))
    with numpyro.plate('n', X_.shape[0]):
        numpyro.sample('y', dist.Categorical(logits=X_ @ beta + alpha), obs=y_)
t0 = time.time()
mcmc = MCMC(NUTS(model_sm, max_tree_depth=10, target_accept_prob=0.8), num_warmup=500, num_samples=500,
            num_chains=2, chain_method='vectorized', progress_bar=False)
mcmc.run(jax.random.PRNGKey(0), Xb, yb, extra_fields=('diverging', 'num_steps'))
nuts_time = time.time() - t0
S = mcmc.get_samples(group_by_chain=True); ex = mcmc.get_extra_fields()
B = np.asarray(S['beta'])
top = np.argsort(-np.abs(B.mean((0, 1))).max(1))[:200]
rh = np.asarray(gelman_rubin(B[:, :, top, :])); ess = np.asarray(effective_sample_size(B[:, :, top, :]))
Bf = B.reshape(-1, P, K); mu, lo, hi = Bf.mean(0), np.percentile(Bf, 3, 0), np.percentile(Bf, 97, 0); sel = (lo > 0) | (hi < 0)
div = int(np.asarray(ex['diverging']).sum())
print(f'NUTS {Nb:,} rows x {P} x {K}: {nuts_time:.0f}s | divergences {div} | mean leapfrog {float(np.asarray(ex["num_steps"]).mean()):.0f} | b {float(np.asarray(S["b"]).mean()):.3f}')
print(f'R-hat top-200: median {np.median(rh):.3f}, max {rh.max():.3f} | ESS median {np.median(ess):.0f} | credible {sel.mean():.2%} ({sel.any(1).sum()} patterns)')
print('P1.7b thresholds: 0 divergences, R-hat max <= 1.05 ->', 'PASS' if div == 0 and rh.max() <= 1.05 else 'FAIL')
np.savez(f'{OUT}/tables_ben.npz', mean=mu, hdi_lo=lo, hdi_hi=hi, selected=sel, patvocab=np.array(PV), classes=np.array(CLASSES),
         n_rows=Nb, divergences=div, rhat_top200_median=float(np.median(rh)), rhat_top200_max=float(rh.max()),
         ess_top200_median=float(np.median(ess)), fit_time_s=nuts_time)

In [ ]:
# R5-pre -- vocabulary coverage of the cached embedding subsets on OSM dev nouns, both backbones
import pipeline_ranker as pr
RK = {bb: pr.load_default(ROOT, backbone=bb) for bb in ('w2v', 'numberbatch')}
dev = D[D.split == 'dev']
nouns = [t for L in dev.context_nouns for t in L if t.lower() not in pr.ABSTRACT]
for bb, R in RK.items():
    cov = np.mean([R.vec(t) is not None for t in nouns])
    rows_any = np.mean([any(R.vec(t) is not None for t in L if t.lower() not in pr.ABSTRACT) for L in dev.context_nouns])
    print(f'{bb:12s} noun tokens embeddable {cov:.1%} | dev rows with >=1 embeddable noun {rows_any:.1%} | tags {len(R.TAGS)}')
print('gold tags outside ranker vocab:', sorted(set(D.tag) - set(RK['w2v'].TAGS)))

In [ ]:
import pickle, glob
obj = pickle.load(open(f'{ROOT}/models_cache/w2v_googlenews_subset_v2.pkl', 'rb'))
k0 = next(iter(obj)); print(type(obj).__name__, len(obj), repr(k0), type(obj[k0]).__name__, getattr(obj[k0], 'shape', None), getattr(obj[k0], 'dtype', None))
for nb in glob.glob('/content/drive/MyDrive/Colab Notebooks/*.ipynb'):
    try: cells = json.load(open(nb))['cells']
    except Exception: continue
    for i, c in enumerate(cells):
        s = ''.join(c.get('source', []))
        if c.get('cell_type') == 'code' and ('numberbatch_subset' in s or 'w2v_googlenews_subset_v2' in s) and ('dump' in s):
            print(f'==== {nb.split("/")[-1]} cell {i} ====\n{s[:1800]}\n')

In [ ]:
# R5-cache-w2v -- M5.1 recipe unchanged, applied to every OSM noun + the lexicon; written to osmfinal/, old caches untouched
import re, gc
LEX = json.load(open(f'{ROOT}/eval/evalset_106.json'))['lexicon']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gensim>=4.3.3'], check=True)
def parts(s): return [w for w in re.split(r'[\s\-_/]+', s) if w]
def variants(s):
    base = [s, s.lower(), s.title(), s.capitalize()]
    joined = [v.replace(' ', '_').replace('-', '_') for v in base]; squashed = [v.replace(' ', '').replace('-', '') for v in base]
    seen, out = set(), []
    for v in base + joined + squashed:
        if v and v not in seen: seen.add(v); out.append(v)
    return out
need = set(LEX)
for L in D.context_nouns:
    for n in L: need.add(n); need.update(parts(n))
OLD = pickle.load(open(f'{ROOT}/models_cache/w2v_googlenews_subset_v2.pkl', 'rb'))
probe = {v for s in need for v in variants(s)} - set(OLD)
import gensim.downloader as api
t0 = time.time(); kv = api.load('word2vec-google-news-300')
NEW = {p: np.asarray(kv[p], dtype=np.float32) for p in probe if p in kv}
del kv; gc.collect()
W2V_OSM = {**OLD, **NEW}
pickle.dump(W2V_OSM, open(f'{OUT}/w2v_googlenews_osm.pkl', 'wb'), protocol=4)
print(f'{len(need):,} strings | +{len(NEW):,} vectors -> {len(W2V_OSM):,} | {time.time()-t0:.0f}s')
for nb in glob.glob('/content/drive/MyDrive/Colab Notebooks/*.ipynb'):
    try: cells = json.load(open(nb))['cells']
    except Exception: continue
    for i, c in enumerate(cells):
        s = ''.join(c.get('source', []))
        if c.get('cell_type') == 'code' and 'numberbatch' in s.lower() and ('.gz' in s or 'conceptnet' in s.lower()):
            print(f'==== {nb.split("/")[-1]} cell {i} ====\n{s[:2500]}\n'); break

In [ ]:
# R5-cache-nb -- P2.1 Numberbatch recipe unchanged on the same OSM string set; then coverage re-measured for both caches
OLDN = pickle.load(open(f'{ROOT}/models_cache/numberbatch_subset.pkl', 'rb'))
probe = {v for s in need for v in variants(s)} - set(OLDN)
t0 = time.time(); kv = api.load('conceptnet-numberbatch-17-06-300')
key = lambda p: f'/c/en/{p.lower()}'
NEWN = {p: np.asarray(kv[key(p)], np.float32) for p in probe if key(p) in kv}
del kv; gc.collect()
NB_OSM = {**OLDN, **NEWN}
pickle.dump(NB_OSM, open(f'{OUT}/numberbatch_osm.pkl', 'wb'), protocol=4)
print(f'numberbatch: +{len(NEWN):,} -> {len(NB_OSM):,} | {time.time()-t0:.0f}s')
RK = {'w2v': pr.Ranker(pr.ladder(W2V_OSM), LEX, pr.Tables(f'{ROOT}/results_i/i3_elasticnet.npz', f'{ROOT}/results_p/p1_bayesian_en.npz', f'{ROOT}/results_p/p1_feat_meta.npz')),
      'numberbatch': pr.Ranker(pr.ladder(NB_OSM), LEX, pr.Tables(f'{ROOT}/results_i/i3_elasticnet.npz', f'{ROOT}/results_p/p1_bayesian_en.npz', f'{ROOT}/results_p/p1_feat_meta.npz'))}
for bb, R in RK.items():
    cov = np.mean([R.vec(t) is not None for t in nouns])
    print(f'{bb:12s} noun tokens embeddable {cov:.1%} | tags {len(R.TAGS)}')

In [ ]:
# R5a -- dev-only selection. Similarity and table scores are computed once per row; the blend and the
# worst-case rank are then vectorised over lambda. Nothing here reads the test split.
T4 = dict(np.load(f'{OUT}/tables_r4a.npz'))
bz = np.load(f'{OUT}/tables_ben.npz'); T4['bayes_en'] = (bz['mean'] * bz['selected']).astype(np.float32)
ARMS = ['conf', 'loglift', 'fen_C0.1', 'fen_C0.5', 'fen_C2.0', 'bayes_en']
LAMS = [round(x, 2) for x in np.arange(0.0, 0.85, 0.05)]
def rows_of(split): return D[D.split == split]
def norm_rows(M):
    lo, hi = M.min(1, keepdims=True), M.max(1, keepdims=True)
    return np.where(hi > lo, (M - lo) / (hi - lo + 1e-9), M)
def sim_matrix(R, df):
    return np.vstack([R._norm(R.sim([t for t in L if t.lower() not in pr.ABSTRACT])) for L in df.context_nouns]).astype(np.float32)
def table_matrix(R, Xs, M):
    agg = Xs @ M                                           # rows x classes, summed over the row's patterns
    S = np.zeros((Xs.shape[0], len(R.TAGS)), np.float32)
    for k, c in enumerate(CLASSES):
        if c in R.IX: S[:, R.IX[c]] = agg[:, k]
    has_ = Xs.getnnz(1) > 0
    S[has_] = norm_rows(S[has_]); S[~has_] = 0.0
    return S
def gold_idx(R, df): return np.array([R.IX[t] for t in df.tag])
def ranks(SIM, TBL, g, lam):
    s = SIM if TBL is None or lam == 0 else (1 - lam) * SIM + lam * TBL
    return (s >= s[np.arange(len(g)), g][:, None]).sum(1)
def mets(r): return {'Hits@1': float((r <= 1).mean()), 'Hits@3': float((r <= 3).mean()), 'nDCG@20': float(np.where(r <= 20, 1 / np.log2(r + 1), 0).mean())}
dv = rows_of('dev'); CACHE = {}
for bb, R in RK.items():
    t0 = time.time(); SIM = sim_matrix(R, dv); g = gold_idx(R, dv)
    CACHE[bb] = {'SIM': SIM, 'g': g, 'TBL': {a: table_matrix(R, X['dev'], T4[a]) for a in ARMS}}
    print(f'{bb}: dev caches built in {time.time()-t0:.0f}s')
DEV = {}
for bb in RK:
    c = CACHE[bb]; DEV[bb] = {'nouns': mets(ranks(c['SIM'], None, c['g'], 0))}
    for a in ARMS:
        curve = {l: mets(ranks(c['SIM'], c['TBL'][a], c['g'], l)) for l in LAMS}
        best = max(LAMS, key=lambda l: (curve[l]['Hits@3'], curve[l]['nDCG@20']))
        DEV[bb][a] = {'best_lambda': best, **curve[best], 'curve_hits3': {str(l): curve[l]['Hits@3'] for l in LAMS}}
    for a, m in DEV[bb].items():
        print(f'{bb:12s} {a:10s} ' + (f"lam {m['best_lambda']:.2f} " if 'best_lambda' in m else '         ') +
              f"H@1 {m['Hits@1']:.3f}  H@3 {m['Hits@3']:.3f}  nDCG {m['nDCG@20']:.3f}")
json.dump(DEV, open(f'{OUT}/r5_dev_selection.json', 'w'), indent=1)

In [ ]:
# R6 -- adoption gate on dev (protocol rule: keep a change only if the paired-bootstrap 95% CI of the
# dev Hits@3 difference excludes zero in its favour). Wikivoyage tables are the current thesis tables.
def hit3(bb, arm, lam=None):
    c = CACHE[bb]
    if arm == 'nouns': return ranks(c['SIM'], None, c['g'], 0) <= 3
    return ranks(c['SIM'], c['TBL'][arm], c['g'], DEV[bb][arm]['best_lambda'] if lam is None else lam) <= 3
dv_rows = [{'adjectives': a, 'transaction': t} for a, t in zip(dv.adjectives, dv.transaction)]
for bb, R in RK.items():                                   # current thesis tables, scored on OSM dev
    CACHE[bb]['TBL']['wv_adj_en'] = np.vstack([R.en_adj(r) for r in dv_rows]).astype(np.float32)
    CACHE[bb]['TBL']['wv_ben']    = np.vstack([R.en_ben(r) for r in dv_rows]).astype(np.float32)
    for a in ('wv_adj_en', 'wv_ben'):
        curve = {l: mets(ranks(CACHE[bb]['SIM'], CACHE[bb]['TBL'][a], CACHE[bb]['g'], l)) for l in LAMS}
        best = max(LAMS, key=lambda l: (curve[l]['Hits@3'], curve[l]['nDCG@20']))
        DEV[bb][a] = {'best_lambda': best, **curve[best]}
        print(f'{bb:12s} {a:10s} lam {best:.2f} H@1 {curve[best]["Hits@1"]:.3f} H@3 {curve[best]["Hits@3"]:.3f} nDCG {curve[best]["nDCG@20"]:.3f}')
def boot(a, b, n=10000, seed=0):
    rs = np.random.RandomState(seed); m = len(a); d = np.empty(n); a = a.astype(float); b = b.astype(float)
    for i in range(n):
        ix = rs.randint(0, m, m); d[i] = a[ix].mean() - b[ix].mean()
    return float(a.mean() - b.mean()), float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))
GATES = [('backbone: numberbatch vs w2v (conf)',            ('numberbatch', 'conf'),     ('w2v', 'conf')),
         ('selection: freq EN C0.5 vs conf (numberbatch)',  ('numberbatch', 'fen_C0.5'), ('numberbatch', 'conf')),
         ('selection: Bayesian EN vs conf (numberbatch)',   ('numberbatch', 'bayes_en'), ('numberbatch', 'conf')),
         ('data: OSM conf vs Wikivoyage adj EN',            ('numberbatch', 'conf'),     ('numberbatch', 'wv_adj_en')),
         ('data: OSM Bayesian EN vs Wikivoyage Bayesian EN',('numberbatch', 'bayes_en'), ('numberbatch', 'wv_ben')),
         ('tables at all: conf vs nouns only',              ('numberbatch', 'conf'),     ('numberbatch', 'nouns'))]
ADOPT = []
for name, A, B in GATES:
    d, lo, hi = boot(hit3(*A), hit3(*B))
    verdict = 'KEEP first' if lo > 0 else ('KEEP second' if hi < 0 else 'NO DIFFERENCE -> simpler/existing stays')
    ADOPT.append({'gate': name, 'a': A, 'b': B, 'delta_hits3': d, 'ci95': [lo, hi], 'verdict': verdict})
    print(f'{name:50s} dH@3 {d:+.3f} [{lo:+.3f},{hi:+.3f}]  {verdict}')
json.dump({'dev_n': len(dv), 'gates': ADOPT, 'dev': DEV}, open(f'{OUT}/adoption.json', 'w'), indent=1)

In [ ]:
# R5b -- learning curves: refit on 10/25/50/100% of train; score (a) up to 7,000 rows drawn from the rows that
# fraction was fitted on, (b) dev, both at the dev-chosen lambda. Converging curves with a small gap = neither
# under- nor over-fitting.
R = RK['numberbatch']; rng = np.random.RandomState(0)
trd = rows_of('train'); SIM_TR = sim_matrix(R, trd); G_TR = gold_idx(R, trd)
yt_all = Y['train']; order = rng.permutation(len(trd)); LC = {}
def conf_from(idx):
    Cn = np.zeros((P, K)); r_, c_ = X['train'][idx].nonzero(); np.add.at(Cn, (c_, yt_all[idx][r_]), 1)
    rt_ = Cn.sum(1, keepdims=True); return np.divide(Cn, rt_, out=np.zeros_like(Cn), where=rt_ > 0).astype(np.float32)
def fen_from(idx):
    m = LogisticRegression(penalty='elasticnet', solver='saga', l1_ratio=0.5, C=0.5, max_iter=200, tol=1e-3, n_jobs=-1).fit(X['train'][idx], yt_all[idx])
    coef = np.zeros((P, K), np.float32); coef[:, m.classes_] = m.coef_.T; return coef
for arm, fit in (('conf', conf_from), ('fen_C0.5', fen_from)):
    lam = DEV['numberbatch'][arm]['best_lambda']; LC[arm] = []
    for f in (0.10, 0.25, 0.50, 1.00):
        idx = np.sort(order[:int(f * len(trd))]); t0 = time.time(); M = fit(idx)
        ev_ = np.sort(rng.choice(idx, min(7000, len(idx)), replace=False))
        h_tr = (ranks(SIM_TR[ev_], table_matrix(R, X['train'][ev_], M), G_TR[ev_], lam) <= 3).mean()
        h_dv = (ranks(CACHE['numberbatch']['SIM'], table_matrix(R, X['dev'], M), CACHE['numberbatch']['g'], lam) <= 3).mean()
        LC[arm].append({'fraction': f, 'n_train': len(idx), 'train_hits3': float(h_tr), 'dev_hits3': float(h_dv), 'gap': float(h_tr - h_dv)})
        print(f'{arm:9s} {f:4.0%} n={len(idx):6,}  train H@3 {h_tr:.3f}  dev H@3 {h_dv:.3f}  gap {h_tr-h_dv:+.3f}  ({time.time()-t0:.0f}s)')
json.dump(LC, open(f'{OUT}/learning_curves.json', 'w'), indent=1)

In [ ]:
import os, time, json, hashlib
print('flag exists:', os.path.exists(f'{OUT}/TEST_SCORED.flag'))

In [ ]:
# R7 -- ONE-SHOT FINAL. Every code path is dry-run on dev first; the flag file is created only after that
# passes and before any test row is scored, and its existence stops a second run. All lambdas dev-chosen.
import hashlib
from scipy.stats import binomtest
FLAG = f'{OUT}/TEST_SCORED.flag'
assert not os.path.exists(FLAG), 'test set already scored once -- protocol forbids a second look'
def gold_idx(R, df): return np.array([R.IX.get(t, -1) for t in df.tag])
def ranks(SIM, TBL, g, lam):                     # gold outside the backbone vocabulary -> worst rank, as Ranker.rank
    s = SIM if TBL is None or lam == 0 else (1 - lam) * SIM + lam * TBL
    ok = g >= 0; r = np.full(len(g), SIM.shape[1] + 1)
    r[ok] = (s[ok] >= s[np.where(ok)[0], g[ok]][:, None]).sum(1); return r
ev106 = json.load(open(f'{ROOT}/eval/evalset_106_extracted.json')); ev106 = ev106['rows'] if isinstance(ev106, dict) else ev106
evpk = json.load(open(f'{ROOT}/eval/pk_evalset_extracted.json')); evpk = evpk['rows'] if isinstance(evpk, dict) else evpk
cnt = Counter(r['gold_tag'] for r in evpk); pk3 = [r for r in evpk if cnt[r['gold_tag']] >= 3]
def as_df(rows):
    return pd.DataFrame({'context_nouns': [r['context_nouns'] for r in rows], 'tag': [r['gold_tag'] for r in rows],
                         'adjs': [sorted({a.lower() for a in r.get('adjectives', []) if str(a).isalpha()}) for r in rows],
                         'pit': [[it for it in r.get('transaction', []) if it.startswith('amod:')] for r in rows],
                         'adjectives': [r.get('adjectives', []) for r in rows], 'transaction': [r.get('transaction', []) for r in rows]})
CFG = {'final: numberbatch + OSM conf': ('numberbatch', 'conf'), 'numberbatch + OSM freq EN': ('numberbatch', 'fen_C0.5'),
       'numberbatch + OSM Bayesian EN': ('numberbatch', 'bayes_en'), 'numberbatch + Wikivoyage adj EN': ('numberbatch', 'wv_adj_en'),
       'w2v + OSM conf': ('w2v', 'conf'), 'base paper (w2v, Algorithm 2)': ('w2v', 'nouns')}
def hits_on(df, bb, arm):
    Rr = RK[bb]; SIM = sim_matrix(Rr, df); g = gold_idx(Rr, df)
    if arm == 'nouns': return ranks(SIM, None, g, 0)
    if arm.startswith('wv_'):
        rows = [{'adjectives': a, 'transaction': t} for a, t in zip(df.adjectives, df.transaction)]
        TB = np.vstack([Rr.en_adj(r) if arm == 'wv_adj_en' else Rr.en_ben(r) for r in rows]).astype(np.float32)
    else:
        TB = table_matrix(Rr, featurize(df), T4[arm])
    return ranks(SIM, TB, g, DEV[bb][arm]['best_lambda'])

# ---- dry run on dev: must reproduce the R5/R6 dev numbers, and every eval frame must build ----
for c, (bb, arm) in CFG.items():
    got = ((hits_on(dv, bb, arm) <= 3).mean()); want = DEV[bb]['nouns' if arm == 'nouns' else arm]['Hits@3']
    assert abs(got - want) < 1e-9, f'dry run mismatch {c}: {got} vs {want}'
for rows in (ev106, pk3):
    f_ = as_df(rows); assert featurize(f_).shape[0] == len(rows)
print('dry run passed: all six configurations reproduce their dev Hits@3 exactly')

te = rows_of('test')
assert hashlib.sha256('\n'.join(sorted(te.osm_id)).encode()).hexdigest() == json.load(open(f'{OUT}/split_stats.json'))['test_sha256_of_sorted_ids'], 'test set differs from the one hashed at split time'
open(FLAG, 'w').write(time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()))
SETS = {'osm_test': te, 'osm_test_pakistan': te[te.pakistan], 'eval_106': as_df(ev106), 'pk_n3': as_df(pk3)}
FINAL = {'sets': {k: len(v) for k, v in SETS.items()}, 'results': {}, 'comparisons': []}; RR = {}
for sname, df in SETS.items():
    RR[sname] = {c: hits_on(df, *a) for c, a in CFG.items()}
    FINAL['results'][sname] = {c: mets(r) for c, r in RR[sname].items()}
    print(f'\n== {sname} (n={len(df)})')
    for c, m in FINAL['results'][sname].items():
        print(f'  {c:36s} H@1 {m["Hits@1"]:.3f}  H@3 {m["Hits@3"]:.3f}  nDCG {m["nDCG@20"]:.3f}')
PRE = [('numberbatch + OSM freq EN', 'final: numberbatch + OSM conf'), ('numberbatch + OSM Bayesian EN', 'final: numberbatch + OSM conf'),
       ('final: numberbatch + OSM conf', 'numberbatch + Wikivoyage adj EN'), ('final: numberbatch + OSM conf', 'w2v + OSM conf'),
       ('final: numberbatch + OSM conf', 'base paper (w2v, Algorithm 2)')]
for sname in SETS:
    for a, b in PRE:
        ha, hb = RR[sname][a] <= 3, RR[sname][b] <= 3
        d, lo, hi = boot(ha, hb); n01, n10 = int((~ha & hb).sum()), int((ha & ~hb).sum())
        p = binomtest(min(n01, n10), n01 + n10, 0.5).pvalue if n01 + n10 else 1.0
        FINAL['comparisons'].append({'set': sname, 'a': a, 'b': b, 'delta_hits3': d, 'ci95': [lo, hi], 'a_only': n10, 'b_only': n01, 'p': float(p)})
ps = np.array([c['p'] for c in FINAL['comparisons']]); o = np.argsort(ps); m_ = len(ps); adj = np.empty(m_); run = 0.0
for rank_, i in enumerate(o): run = max(run, min(1.0, (m_ - rank_) * ps[i])); adj[i] = run
for c, pa in zip(FINAL['comparisons'], adj): c['p_holm'] = float(pa)
print('\n== pre-registered comparisons, Hits@3, Holm across', m_, 'tests')
for c in FINAL['comparisons']:
    print(f"  {c['set']:18s} {c['a'][:30]:30s} vs {c['b'][:30]:30s} dH@3 {c['delta_hits3']:+.3f} [{c['ci95'][0]:+.3f},{c['ci95'][1]:+.3f}] p_holm {c['p_holm']:.4f}")
json.dump(FINAL, open(f'{OUT}/final.json', 'w'), indent=1)
print('\nwrote', f'{OUT}/final.json')

In [ ]:
# FIGURES + TABLES -- every number read from the osmfinal result files or the kernel's final ranks; nothing retyped.
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
FIG, TAB = f'{OUT}/figures', f'{OUT}/tables'; os.makedirs(FIG, exist_ok=True); os.makedirs(TAB, exist_ok=True)
plt.rcParams.update({'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False, 'savefig.dpi': 300, 'savefig.bbox': 'tight'})
INK, GREY, LIGHT = '#000000', '#6E6E6E', '#C8C8C8'
FIN = json.load(open(f'{OUT}/final.json')); ADP = json.load(open(f'{OUT}/adoption.json')); LCJ = json.load(open(f'{OUT}/learning_curves.json'))
SPL = json.load(open(f'{OUT}/split_stats.json')); DEVJ = json.load(open(f'{OUT}/r5_dev_selection.json')); FUN = json.load(open(f'{OUT}/funnel.json'))
def save(fig, name): fig.savefig(f'{FIG}/{name}.png'); fig.savefig(f'{FIG}/{name}.pdf'); plt.close(fig); print('fig', name)
def ci(h, n=10000, seed=0):
    rs = np.random.RandomState(seed); h = h.astype(float); b = [h[rs.randint(0, len(h), len(h))].mean() for _ in range(n)]
    return np.percentile(b, 2.5), np.percentile(b, 97.5)
SHORT = {'final: numberbatch + OSM conf': 'Final\n(NB + OSM assoc.)', 'numberbatch + OSM freq EN': 'NB + OSM\nfreq. EN',
         'numberbatch + OSM Bayesian EN': 'NB + OSM\nBayesian EN', 'numberbatch + Wikivoyage adj EN': 'NB + Wikivoyage\nEN (previous)',
         'w2v + OSM conf': 'W2V + OSM\nassoc.', 'base paper (w2v, Algorithm 2)': 'Base paper\n(W2V, Alg. 2)'}

# F1 held-out test, Hits@3 with 95% CI
cfgs = list(CFG); h = [(RR['osm_test'][c] <= 3) for c in cfgs]; m = [x.mean() for x in h]; cis = [ci(x) for x in h]
fig, ax = plt.subplots(figsize=(9, 4.2))
cols = [INK if c.startswith('final') else (LIGHT if c.startswith('base') else GREY) for c in cfgs]
ax.bar(range(len(cfgs)), m, color=cols, width=0.62, yerr=[[a - l for a, (l, _) in zip(m, cis)], [u - a for a, (_, u) in zip(m, cis)]], capsize=4, ecolor=INK)
for i, v in enumerate(m): ax.text(i, v + 0.02, f'{v:.3f}', ha='center')
ax.set_xticks(range(len(cfgs))); ax.set_xticklabels([SHORT[c] for c in cfgs], fontsize=9); ax.set_ylabel('Hits@3'); ax.set_ylim(0, max(m) * 1.25)
ax.set_title(f'Held-out OSM test set (n = {FIN["sets"]["osm_test"]:,}), Hits@3 with 95% bootstrap CI', loc='left')
save(fig, 'fig_4.20_heldout_test_hits3')

# F2 all evaluation sets, three key configurations
keys = ['base paper (w2v, Algorithm 2)', 'numberbatch + OSM freq EN', 'final: numberbatch + OSM conf']
sets = ['osm_test', 'eval_106', 'pk_n3']; lab = {'osm_test': f'OSM test\nn={FIN["sets"]["osm_test"]:,}', 'eval_106': f'Base-paper set\nn={FIN["sets"]["eval_106"]}', 'pk_n3': f'Pakistan n≥3\nn={FIN["sets"]["pk_n3"]}'}
fig, ax = plt.subplots(figsize=(9, 4.2)); w = 0.26
for j, (k, c) in enumerate(zip(keys, [LIGHT, GREY, INK])):
    vals = [FIN['results'][s][k]['Hits@3'] for s in sets]
    ax.bar(np.arange(len(sets)) + (j - 1) * w, vals, w, color=c, label=SHORT[k].replace('\n', ' '), edgecolor=INK, linewidth=0.5)
    for i, v in enumerate(vals): ax.text(i + (j - 1) * w, v + 0.012, f'{v:.2f}', ha='center', fontsize=8)
ax.set_xticks(range(len(sets))); ax.set_xticklabels([lab[s] for s in sets]); ax.set_ylabel('Hits@3'); ax.legend(frameon=False, fontsize=9)
ax.set_title('Same configurations on every evaluation set (dev-chosen λ, no tuning on any test set)', loc='left')
save(fig, 'fig_4.21_all_sets_hits3')

# F3 learning curves
fig, axs = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
for ax, arm, t in zip(axs, ['conf', 'fen_C0.5'], ['Association table (no selection)', 'Frequentist elastic net']):
    n = [r['n_train'] for r in LCJ[arm]]
    ax.plot(n, [r['train_hits3'] for r in LCJ[arm]], '-o', color=INK, label='train'); ax.plot(n, [r['dev_hits3'] for r in LCJ[arm]], '--s', color=GREY, label='dev')
    for r in LCJ[arm]: ax.annotate(f"gap {r['gap']:+.3f}", (r['n_train'], r['dev_hits3']), textcoords='offset points', xytext=(0, -14), ha='center', fontsize=8)
    ax.set_xscale('log'); ax.set_xlabel('training rows'); ax.set_title(t, loc='left')
axs[0].set_ylabel('Hits@3'); axs[0].legend(frameon=False); fig.suptitle('Learning curves: the train–dev gap closes as data grows', x=0.01, ha='left')
save(fig, 'fig_4.22_learning_curves')

# F4 adoption gate forest plot
g = ADP['gates']; fig, ax = plt.subplots(figsize=(9, 3.8))
for i, x in enumerate(g):
    lo, hi = x['ci95']; ax.plot([lo, hi], [i, i], color=INK, lw=1.6); ax.plot(x['delta_hits3'], i, 'o', color=INK)
ax.axvline(0, color=GREY, lw=1, ls=':'); ax.set_yticks(range(len(g))); ax.set_yticklabels([x['gate'] for x in g], fontsize=9); ax.invert_yaxis()
ax.set_xlabel('Δ Hits@3 on dev (first − second), 95% paired-bootstrap CI'); ax.set_title(f'Adoption gate on dev (n = {ADP["dev_n"]:,})', loc='left')
save(fig, 'fig_4.23_adoption_gate')

# F5 lambda curves on dev
fig, ax = plt.subplots(figsize=(8, 3.8)); styles = {'conf': ('-', INK), 'fen_C0.5': ('--', INK), 'bayes_en': (':', INK), 'loglift': ('-', GREY)}
names = {'conf': 'association (no selection)', 'fen_C0.5': 'freq. EN', 'bayes_en': 'Bayesian EN', 'loglift': 'log-lift (no selection)'}
for a, (ls, c) in styles.items():
    cv = DEVJ['numberbatch'][a]['curve_hits3']; x = [float(k) for k in cv]; ax.plot(x, list(cv.values()), ls=ls, color=c, label=names[a])
ax.axhline(DEVJ['numberbatch']['nouns']['Hits@3'], color=LIGHT, lw=1); ax.text(0.8, DEVJ['numberbatch']['nouns']['Hits@3'] + 0.005, 'nouns only', ha='right', fontsize=8, color=GREY)
ax.set_xlabel('blend weight λ'); ax.set_ylabel('dev Hits@3'); ax.legend(frameon=False, fontsize=9); ax.set_title('Blend weight chosen on dev (Numberbatch backbone)', loc='left')
save(fig, 'fig_4.24_dev_lambda_curves')

# F6 corpus funnel, read from funnel.json (written by the FUNNEL cell above)
v = [x['rows'] for x in FUN]
fig, ax = plt.subplots(figsize=(8, 3.4)); ax.barh(range(len(FUN)), v, color=[GREY] * (len(FUN) - 1) + [INK]); ax.set_xscale('log'); ax.invert_yaxis()
ax.set_yticks(range(len(FUN))); ax.set_yticklabels([x['step'] for x in FUN], fontsize=9)
for i, x in enumerate(v): ax.text(x * 1.05, i, f'{x:,}', va='center', fontsize=9)
ax.set_xlim(right=max(v) * 4); ax.set_title('From OpenStreetMap descriptions to the evaluation corpus', loc='left')
save(fig, 'fig_3.10_osm_corpus_funnel')

# TABLES
rows = [{'set': s, 'n': FIN['sets'][s], 'configuration': c, **{k: round(v, 3) for k, v in m_.items()}} for s, d in FIN['results'].items() for c, m_ in d.items()]
t1 = pd.DataFrame(rows); t1.to_csv(f'{TAB}/tab_4.20_final_results.csv', index=False); t1.to_latex(f'{TAB}/tab_4.20_final_results.tex', index=False)
t2 = pd.DataFrame([{'set': c['set'], 'comparison': f"{c['a']} vs {c['b']}", 'dHits3': round(c['delta_hits3'], 3),
                    'ci95': f"[{c['ci95'][0]:+.3f}, {c['ci95'][1]:+.3f}]", 'p': round(c['p'], 4), 'p_holm': round(c['p_holm'], 4)} for c in FIN['comparisons']])
t2.to_csv(f'{TAB}/tab_4.21_prespecified_comparisons.csv', index=False); t2.to_latex(f'{TAB}/tab_4.21_prespecified_comparisons.tex', index=False)
t3 = pd.DataFrame([{'split': k, 'rows': v, 'pakistan': SPL['pakistan_by_split'].get(k, 0)} for k, v in SPL['split_counts'].items()])
t3.to_csv(f'{TAB}/tab_3.10_osm_split.csv', index=False); t3.to_latex(f'{TAB}/tab_3.10_osm_split.tex', index=False)
t4 = pd.DataFrame([{'arm': a, 'fraction': r['fraction'], 'n_train': r['n_train'], 'train_hits3': round(r['train_hits3'], 3), 'dev_hits3': round(r['dev_hits3'], 3), 'gap': round(r['gap'], 3)} for a, L_ in LCJ.items() for r in L_])
t4.to_csv(f'{TAB}/tab_4.22_learning_curves.csv', index=False); t4.to_latex(f'{TAB}/tab_4.22_learning_curves.tex', index=False)
t5 = pd.DataFrame(FUN); t5.to_csv(f'{TAB}/tab_3.11_osm_corpus_funnel.csv', index=False); t5.to_latex(f'{TAB}/tab_3.11_osm_corpus_funnel.tex', index=False)
print('tables:', sorted(os.listdir(TAB)))

In [ ]:
# FUNNEL -- recompute every corpus-funnel count from the raw harvest with the R1b/R1c rules unchanged; writes
# funnel.json (the figure cell draws from it). Reads nothing from the test split; writes no corpus file.
# Check: the last count must equal split_stats.json 'rows', or the recomputation differs from the original.
import re, json, pandas as pd, py3langid.langid as L
SPL = json.load(open(f'{OUT}/split_stats.json')); LEX = json.load(open(f'{ROOT}/eval/evalset_106.json'))['lexicon']
raw = pd.read_parquet(f'{OUT}/raw/osm_descriptions.parquet'); LEXSET = set(LEX)
norm = lambda s: s.lower().replace('_', '-').replace(' ', '-') if isinstance(s, str) else None
raw['tag'] = raw['value'].map(norm).where(lambda v: v.isin(LEXSET))
EXTRA = {('highway', 'cycleway'): 'cycle-way', ('highway', 'footway'): 'foot-way', ('natural', 'wetland'): 'wet-land', ('natural', 'wood'): 'woods',
         ('natural', 'sinkhole'): 'sink-hole', ('amenity', 'arts_centre'): 'arts-center', ('building', 'greenhouse'): 'green-house',
         ('building', 'houseboat'): 'house-boat', ('man_made', 'wastewater_plant'): 'waste-water', ('shop', 'books'): 'bookshop',
         ('place', 'province'): 'province', ('place', 'ocean'): 'ocean', ('place', 'continent'): 'continent'}
RELIGION = {'muslim': 'mosque', 'christian': 'church', 'buddhist': 'temple', 'hindu': 'temple', 'jain': 'temple', 'sikh': 'temple', 'shinto': 'temple', 'taoist': 'temple'}
EXTRA = {k: t for k, t in EXTRA.items() if t in LEXSET}; RELIGION = {r: t for r, t in RELIGION.items() if t in LEXSET}
raw['tag'] = raw['tag'].fillna(pd.Series([EXTRA.get(x) for x in zip(raw['key'], raw['value'])], index=raw.index))
pw = (raw['key'] == 'amenity') & (raw['value'] == 'place_of_worship'); raw.loc[pw, 'tag'] = raw.loc[pw, 'religion'].map(RELIGION)
m = raw[raw['tag'].notna()]; nt = m.groupby('osm_id')['tag'].nunique()
el = m[m['osm_id'].isin(nt[nt == 1].index)].drop_duplicates('osm_id')[['osm_id', 'tag', 'key', 'value', 'description', 'name', 'pakistan']]
lid = L.LanguageIdentifier.from_model_file(L.MODEL_FILE, norm_probs=True)
d = el.copy(); d['description'] = d['description'].str.replace(r'\s+', ' ', regex=True).str.strip()
pred = [lid.classify(t) for t in d['description'].tolist()]
en = d[[p[0] == 'en' and p[1] >= 0.6 for p in pred]].copy(); n_en = len(en)
def mask_pattern(tag, value):
    phrases = {tag.replace('-', ' '), str(value).replace('_', ' ')}
    alts = sorted({re.escape(p).replace(r'\ ', r'[\s_-]?') for p in phrases if p}, key=len, reverse=True)
    return re.compile(r'\b(?:' + '|'.join(alts) + r')(?:e?s)?\b', re.I)
PAT = {}
def mask(r):
    k = (r.tag, r.value)
    if k not in PAT: PAT[k] = mask_pattern(*k)
    return PAT[k].sub(' ', r.description)
en['text'] = pd.Series([mask(r) for r in en.itertuples(index=False)], index=en.index).str.replace(r'\s+', ' ', regex=True).str.strip()
en = en[en['text'].str.count(r'\b[A-Za-z]{2,}\b') >= 4]; n_words = len(en)
en['k'] = en['text'].str.lower().str.replace(r'\d+', '#', regex=True).str.replace(r'[^a-z# ]', '', regex=True); en = en.drop_duplicates('k')
CAP, MIN_ = SPL['cap_per_tag'], SPL['min_per_tag']; en = en.sample(frac=1.0, random_state=0)
cp = pd.concat([pd.concat([g[g.pakistan], g[~g.pakistan].head(max(0, CAP - int(g.pakistan.sum())))]) for _, g in en.groupby('tag')])
vc = cp['tag'].value_counts(); cp = cp[cp['tag'].isin(vc[vc >= MIN_].index)]
FUNNEL = [('OSM elements with a description (harvest rows)', len(raw)), ('mapped to the lexicon, unambiguous', len(el)),
          ('English (p ≥ .6)', n_en), ('≥ 4 content words after masking', n_words),
          ('after template dedup', len(en)), (f'capped at {CAP}/tag, ≥ {MIN_} per tag', len(cp))]
assert FUNNEL[-1][1] == SPL['rows'], f'recomputed {FUNNEL[-1][1]} != split_stats {SPL["rows"]}'
json.dump([{'step': s, 'rows': int(n)} for s, n in FUNNEL], open(f'{OUT}/funnel.json', 'w'), indent=1)
for s, n in FUNNEL: print(f'{n:>10,}  {s}')

In [ ]:
# SLIDES -- 16:9, white ground, black type, one figure per slide with the finding stated in one line
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'python-pptx'], check=True)
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
SL = f'{OUT}/slides'; os.makedirs(SL, exist_ok=True)
prs = Presentation(); prs.slide_width, prs.slide_height = Inches(13.333), Inches(7.5); BL = prs.slide_layouts[6]
def text(s, t, x, y, w, h, size, bold=False, color=(0, 0, 0)):
    tb = s.shapes.add_textbox(Inches(x), Inches(y), Inches(w), Inches(h)); tf = tb.text_frame; tf.word_wrap = True
    for i, line in enumerate(t.split('\n')):
        p = tf.paragraphs[0] if i == 0 else tf.add_paragraph(); r = p.add_run(); r.text = line
        r.font.size = Pt(size); r.font.bold = bold; r.font.name = 'Helvetica'; r.font.color.rgb = RGBColor(*color)
def fig_slide(title, finding, png):
    s = prs.slides.add_slide(BL); text(s, title, 0.6, 0.35, 12, 0.8, 30, True)
    text(s, finding, 0.6, 1.15, 12, 0.8, 16, color=(0x55, 0x55, 0x55)); s.shapes.add_picture(f'{FIG}/{png}.png', Inches(1.2), Inches(2.0), height=Inches(5.1))
r = FIN['results']; fo = r['osm_test']['final: numberbatch + OSM conf']['Hits@3']; bo = r['osm_test']['base paper (w2v, Algorithm 2)']['Hits@3']
cmp_ = {(c['set'], c['a'], c['b']): c for c in FIN['comparisons']}
fb = cmp_[('osm_test', 'final: numberbatch + OSM conf', 'base paper (w2v, Algorithm 2)')]
s = prs.slides.add_slide(BL)
text(s, 'Geospatial Tagging of Volunteered Place Descriptions', 0.8, 2.3, 11.5, 1.2, 38, True)
text(s, 'Final pre-registered evaluation on OpenStreetMap descriptions', 0.8, 3.5, 11.5, 0.8, 22, color=(0x55, 0x55, 0x55))
text(s, 'Faizan Shah Ahmad · MS Artificial Intelligence · Air University Islamabad', 0.8, 5.6, 11.5, 0.6, 16)
s = prs.slides.add_slide(BL); text(s, 'Protocol, fixed before any data was seen', 0.6, 0.35, 12, 0.8, 30, True)
text(s, ('• Primary metric Hits@3; worst-case tie rule\n'
         f'• Split by OSM element and name: train {SPL["split_counts"]["train"]:,} · dev {SPL["split_counts"]["dev"]:,} · test {SPL["split_counts"]["test"]:,}\n'
         '• Every hyperparameter (λ, C, backbone) chosen on dev only\n'
         '• A change is kept only if its dev 95% CI excludes zero\n'
         '• Test set hashed at split time and scored exactly once\n'
         '• Pre-registered comparisons, Holm-corrected'), 0.8, 1.5, 11.5, 5, 22)
fig_slide('Data: OpenStreetMap descriptions', f'{SPL["rows"]:,} English descriptions, {SPL["tags"]} tags, type words masked', 'fig_3.10_osm_corpus_funnel')
fig_slide('Hyperparameters chosen on dev only', 'Every table beats nouns-only similarity; λ picked on dev', 'fig_4.24_dev_lambda_curves')
fig_slide('Adoption gate', 'Numberbatch kept; unselected association beats both elastic nets on accuracy', 'fig_4.23_adoption_gate')
fig_slide('Neither under- nor over-fitting', f'Train–dev gap closes to {LCJ["conf"][-1]["gap"]:.3f} at full data; dev still rising', 'fig_4.22_learning_curves')
fig_slide('Held-out test: the result', f'Hits@3 {bo:.3f} → {fo:.3f} over the base-paper method (Δ {fb["delta_hits3"]:+.3f}, Holm p {fb["p_holm"]:.1e})', 'fig_4.20_heldout_test_hits3')
fig_slide('Same direction on every evaluation set', 'Base-paper and Pakistan sets agree in direction; too small for significance after Holm', 'fig_4.21_all_sets_hits3')
s = prs.slides.add_slide(BL); text(s, 'What the evidence supports', 0.6, 0.35, 12, 0.8, 30, True)
text(s, (f'• The pipeline doubles the base-paper method on {FIN["sets"]["osm_test"]:,} held-out descriptions\n'
         '• Mined adjective→tag association carries the gain; Numberbatch beats Word2Vec\n'
         '• Elastic-net selection does not add accuracy — it earns sparse, readable weights\n'
         '• The Bayesian elastic net earns calibrated credible intervals: explanations without post-hoc XAI (RQ3)\n'
         '• Base-paper and Pakistan sets: same direction, underpowered after correction'), 0.8, 1.5, 11.8, 5, 22)
out = f'{SL}/final_run_osm.pptx'; prs.save(out); print('wrote', out, len(prs.slides), 'slides')

In [ ]:
# POST-HOC source -- print the frozen clustered (Algorithm 3, reading B) ranking code verbatim
src = ''.join(json.load(open('/content/drive/MyDrive/Colab Notebooks/sprint_block_l.ipynb'))['cells'][9]['source'])
i = src.index('def ranks_with'); print(src[i:i + 1600])

In [ ]:
# POST-HOC AUDITS (not pre-registered). Writes osmfinal/posthoc_audits.json only; final.json and the test flag are untouched.
from itertools import groupby
from collections import Counter
ABSTRACT = pr.ABSTRACT; LAM = 0.0
exec(src[src.index('def ranks_with'):src.index('def cv_with')])          # frozen Algorithm 3 code, verbatim
te = rows_of('test'); trn = rows_of('train'); dvn = rows_of('dev')
OUTP = {'status': 'post-hoc, not pre-registered', 'test_flag_written': open(f'{OUT}/TEST_SCORED.flag').read()}

# C -- near-duplicates across splits (token Jaccard >= .6, the P2.6 rule)
tok = lambda s: set(re.findall(r'[a-z]{3,}', s.lower()))
trT = [tok(t) for t in trn.text]; inv = {}
for i, s in enumerate(trT):
    for w in s: inv.setdefault(w, []).append(i)
def has_near(t):
    q_ = tok(t)
    if not q_: return False
    c = Counter(i for w in q_ for i in inv.get(w, []))
    return any(n / len(q_ | trT[i]) >= 0.6 for i, n in c.items())
te_dup = np.array([has_near(t) for t in te.text]); dv_dup = np.array([has_near(t) for t in dvn.text])
fin = RR['osm_test']['final: numberbatch + OSM conf'] <= 3; base = RR['osm_test']['base paper (w2v, Algorithm 2)'] <= 3
OUTP['C_near_duplicates'] = {'test_rows_with_train_near_dup': int(te_dup.sum()), 'test_n': len(te), 'dev_rows_with_train_near_dup': int(dv_dup.sum()),
                             'test_hits3_excluding_near_dups': {'final': float(fin[~te_dup].mean()), 'base_alg2': float(base[~te_dup].mean())}}
print('C', OUTP['C_near_duplicates'])

# D -- cue leakage: test rows whose masked text still contains a WordNet noun synonym of the gold tag
import nltk; nltk.download('wordnet', quiet=True); from nltk.corpus import wordnet as wn
def syns(tag):
    out = set()
    for s in wn.synsets(tag.replace('-', '_'), pos='n'):
        out |= {l.name().replace('_', ' ').lower() for l in s.lemmas()}
    return out - {tag.replace('-', ' ')}
SY = {t: syns(t) for t in te.tag.unique()}
leak = np.array([any(re.search(r'\b' + re.escape(w) + r's?\b', t.lower()) for w in SY[g]) for t, g in zip(te.text, te.tag)])
OUTP['D_cue_leak'] = {'definition': 'masked text contains a WordNet noun-synonym lemma of the gold tag (whole word, optional s)',
                      'test_rows_leaking': int(leak.sum()), 'rate': float(leak.mean()),
                      'test_hits3_non_leaking': {'final': float(fin[~leak].mean()), 'base_alg2': float(base[~leak].mean())},
                      'examples': [(g, t[:120]) for t, g, l in zip(te.text, te.tag, leak) if l][:8]}
print('D', {k: v for k, v in OUTP['D_cue_leak'].items() if k != 'examples'})

# E -- base paper's best (Algorithm 3, clustered) on OSM test. Gate: must reproduce .462 Hits@3 on the 106 set first.
rk106 = pr.load_default(ROOT, backbone='w2v')
g106 = (ranks_with(rk106, ev106, clustered=True) <= 3).mean()
assert abs(g106 - 0.462) < 0.006, f'clustered reproduction gate failed: {g106:.3f}'
te_rows = [{'context_nouns': L, 'adjectives': [], 'gold_tag': g} for L, g in zip(te.context_nouns, te.tag)]
alg3 = ranks_with(RK['w2v'], te_rows, clustered=True) <= 3
d_, lo_, hi_ = boot(fin, alg3)
OUTP['E_alg3_on_osm_test'] = {'gate_106_hits3': float(g106), 'alg3_hits3': float(alg3.mean()), 'alg2_hits3': float(base.mean()),
                              'final_minus_alg3': d_, 'ci95': [lo_, hi_]}
print('E', OUTP['E_alg3_on_osm_test'])
json.dump(OUTP, open(f'{OUT}/posthoc_audits.json', 'w'), indent=1, default=str)
print('wrote posthoc_audits.json')

In [ ]:
# POST-HOC E, rerun: Algorithm 3 two ways, each gated on its own known 106-set figure.
#   faithful   -- the base paper's method, no abstract-noun removal     -> gate .462 (base-paper reproduction)
#   with_abs   -- plus our abstract-noun removal (as the Alg. 2 row in final.json) -> gate .491 (the H5 arm)
te_rows = [{'context_nouns': L, 'adjectives': [], 'gold_tag': g} for L, g in zip(te.context_nouns, te.tag)]
E = {}
for name, absset, want in (('faithful', set(), 0.462), ('with_abstract_removal', pr.ABSTRACT, 0.491)):
    ABSTRACT = absset
    g106 = (ranks_with(rk106, ev106, clustered=True) <= 3).mean()
    assert abs(g106 - want) < 0.006, f'{name}: gate {g106:.3f} != {want}'
    a3 = ranks_with(RK['w2v'], te_rows, clustered=True) <= 3
    d_, lo_, hi_ = boot(fin, a3)
    E[name] = {'gate_106_hits3': float(g106), 'alg3_osm_test_hits3': float(a3.mean()), 'final_minus_alg3': d_, 'ci95': [lo_, hi_]}
    print(name, E[name])
ABSTRACT = pr.ABSTRACT
OUTP['E_alg3_on_osm_test'] = {**E, 'alg2_osm_test_hits3_in_final': float(base.mean()),
                              'note': 'the Alg. 2 base row in final.json applied abstract-noun removal, which favours the base method'}
json.dump(OUTP, open(f'{OUT}/posthoc_audits.json', 'w'), indent=1, default=str)
print('wrote posthoc_audits.json')

In [ ]:
def show(p, depth=2):
    d = json.load(open(f'{ROOT}/{p}'))
    def walk(x, pre='', lvl=0):
        if isinstance(x, dict) and lvl < depth:
            for k, v in list(x.items())[:14]: walk(v, f'{pre}.{k}' if pre else k, lvl + 1)
        else: print(f'  {pre}: {str(x)[:110]}')
    print('==', p); walk(d)
for p in ['results_h/h3_reproduction.json', 'results_i/j1_lambda_selection.json', 'results_m/m5_transfer.json',
          'results_p/p1_ben_figure2.json', 'results_i/i2_guided_vs_extended.json']:
    try: show(p)
    except Exception as e: print('==', p, 'ERR', e)
print(sorted(os.listdir(f'{ROOT}/results_i')))

In [ ]:
# DEFENCE DECK -- what we did, how, the final OSM run, and the gain over the base paper. Every number is read from a result file.
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
from pptx.enum.shapes import MSO_SHAPE
from pptx.enum.text import PP_ALIGN, MSO_ANCHOR
H3 = json.load(open(f'{ROOT}/results_h/h3_reproduction.json')); J1 = json.load(open(f'{ROOT}/results_i/j1_lambda_selection.json'))
M5 = json.load(open(f'{ROOT}/results_m/m5_transfer.json')); BEN = json.load(open(f'{ROOT}/results_p/p1_ben_figure2.json'))
FIN = json.load(open(f'{OUT}/final.json')); ADP = json.load(open(f'{OUT}/adoption.json')); PH = json.load(open(f'{OUT}/posthoc_audits.json'))
LCJ = json.load(open(f'{OUT}/learning_curves.json')); SPL = json.load(open(f'{OUT}/split_stats.json')); FUN = json.load(open(f'{OUT}/funnel.json'))
BK, GR, LG, WH = RGBColor(0, 0, 0), RGBColor(0x6E, 0x6E, 0x6E), RGBColor(0xD9, 0xD9, 0xD9), RGBColor(0xFF, 0xFF, 0xFF)
prs = Presentation(); prs.slide_width, prs.slide_height = Inches(13.333), Inches(7.5); BL = prs.slide_layouts[6]
def T(s, t, x, y, w, h, size, bold=False, color=BK, align=PP_ALIGN.LEFT):
    tb = s.shapes.add_textbox(Inches(x), Inches(y), Inches(w), Inches(h)); tf = tb.text_frame; tf.word_wrap = True
    for i, line in enumerate(t.split('\n')):
        p = tf.paragraphs[0] if i == 0 else tf.add_paragraph(); p.alignment = align; r = p.add_run(); r.text = line
        r.font.size = Pt(size); r.font.bold = bold; r.font.name = 'Helvetica'; r.font.color.rgb = color
    return tb
def box(s, x, y, w, h, fill, line=BK, text='', size=14, fc=BK, bold=False):
    sh = s.shapes.add_shape(MSO_SHAPE.RECTANGLE, Inches(x), Inches(y), Inches(w), Inches(h))
    sh.fill.solid(); sh.fill.fore_color.rgb = fill; sh.line.color.rgb = line; sh.line.width = Pt(1.25); sh.shadow.inherit = False
    tf = sh.text_frame; tf.word_wrap = True; tf.vertical_anchor = MSO_ANCHOR.MIDDLE
    for i, line_ in enumerate(text.split('\n')):
        p = tf.paragraphs[0] if i == 0 else tf.add_paragraph(); p.alignment = PP_ALIGN.CENTER; r = p.add_run(); r.text = line_
        r.font.size = Pt(size); r.font.name = 'Helvetica'; r.font.color.rgb = fc; r.font.bold = bold
    return sh
def arrow(s, x, y, w):
    a = s.shapes.add_shape(MSO_SHAPE.RIGHT_ARROW, Inches(x), Inches(y), Inches(w), Inches(0.28)); a.fill.solid(); a.fill.fore_color.rgb = BK; a.line.fill.background()
STAGES = ['Extract\nnouns + adjectives', 'Mine\nadjective–noun patterns', 'Select\nBayesian elastic net', 'Rank\nblend into OSM tags', 'Evaluate\nHits@3']
def strip(s, active=None, y=6.35):
    w, gap, x = 2.25, 0.3, 0.75
    for i, st in enumerate(STAGES):
        on = active == i; box(s, x, y, w, 0.75, BK if on else WH, BK if on else LG, f'{i+1}  {st.split(chr(10))[0]}', 12, WH if on else (BK if active is None else GR), on)
        if i < len(STAGES) - 1: arrow(s, x + w + 0.03, y + 0.24, gap - 0.06)
        x += w + gap
def slide(title, sub=None):
    s = prs.slides.add_slide(BL); T(s, title, 0.75, 0.45, 11.8, 1.0, 30, True)
    if sub: T(s, sub, 0.75, 1.35, 11.8, 0.6, 16, color=GR)
    return s
def bars(s, items, x, y, w, h, maxv, fmt='{:.3f}'):
    n = len(items); bw = w / n * 0.62
    for i, (lab, v, strong) in enumerate(items):
        bh = h * v / maxv; bx = x + i * w / n + (w / n - bw) / 2
        box(s, bx, y + h - bh, bw, bh, BK if strong else LG, BK)
        T(s, fmt.format(v), bx - 0.3, y + h - bh - 0.45, bw + 0.6, 0.4, 16, True, align=PP_ALIGN.CENTER)
        T(s, lab, bx - 0.45, y + h + 0.08, bw + 0.9, 0.8, 12, color=GR, align=PP_ALIGN.CENTER)

r106b, r106o = H3['W2V_clustered']['Hits@3'], J1['cv_held_out']['Hits@3']
pkb, pko = M5['n3_158']['W2V (Alg.2)']['Hits@3'], M5['n3_158']['transfer v1, λ by 5-fold CV']['Hits@3']
F = FIN['results']; K_FIN, K_BASE = 'final: numberbatch + OSM conf', 'base paper (w2v, Algorithm 2)'
cmp_ = {(c['set'], c['a'], c['b']): c for c in FIN['comparisons']}; fb = cmp_[('osm_test', K_FIN, K_BASE)]

# 1 title
s = prs.slides.add_slide(BL)
T(s, 'Geospatial Tagging of Volunteered Place Descriptions', 0.75, 2.2, 11.8, 1.2, 40, True)
T(s, 'Bayesian Elastic Net and Extended FP-Growth Approach', 0.75, 3.3, 11.8, 0.8, 24, color=GR)
T(s, 'Faizan Shah Ahmad (222380)\nMS Artificial Intelligence, Air University Islamabad\nSupervisor Dr. Madiha Yousaf  ·  Co-supervisor Dr. Faridoon Khan', 0.75, 5.2, 11.8, 1.3, 15)

# 2 the gap
s = slide('The base paper reads the nouns and throws the adjectives away', 'Yousaf, Schwartz & Wolter (2023): a description is reduced to its nouns before it is matched to OSM tags')
T(s, '“This parsian place on Elizabeth Street\nmakes some of the best bread”', 0.75, 2.4, 7.2, 1.6, 26)
box(s, 0.75, 4.35, 3.4, 0.7, WH, BK, 'base paper reads:  place, bread', 14)
box(s, 4.45, 4.35, 3.5, 0.7, BK, BK, 'we also read:  parsian, best', 14, WH, True)
T(s, 'Footnote 6 of the base paper: attribute adjectives are ignored.\n\nAdjectives carry the type: sandy → beach, islamic → mosque,\nbotanical → garden. The thesis mines them, selects them, and\nadds them to the ranking.', 8.4, 2.4, 4.4, 3.2, 16, color=GR)

# 3 pipeline overview
s = slide('One pipeline, five stages', 'Every stage is named in the proposal; each one is measured on its own')
DESC = ['spaCy transformer\n(en_core_web_trf)\npulls context nouns\nand adjectives', 'guided extended\nFP-Growth mines\nadjective and\nadjective–noun patterns',
        'Bayesian elastic net\nweights each pattern\nper tag, with a 94%\ncredible interval', 'embedding similarity\nof the nouns, blended\nwith the pattern table\nby weight λ',
        'Hits@k and nDCG@20\nagainst gold OSM tags,\nworst-case ties']
for i, d in enumerate(DESC): T(s, d, 0.75 + i * 2.55, 2.6, 2.3, 2.4, 14, align=PP_ALIGN.CENTER)
strip(s, None, y=1.9)

# 4-8 one stage each
ST = [('Stage 1: extract the words that describe the place', 'A transformer parser finds nouns and the adjectives attached to them',
       'Recall of the base paper\'s own nouns: 73%.\nAdjectives found in 95% of their 106 sentences,\nall of them discarded by the base method.'),
      ('Stage 2: mine which adjectives go with which tags', 'Guided extended FP-Growth over ~400k weakly labelled sentences',
       'Guided and extended FP-Growth return the same rules.\nGuided runs 2,184× faster and scales to the full corpus,\nwhere extended did not finish.'),
      ('Stage 3: select and weight the patterns', 'Bayesian elastic net, Li & Lin scale-mixture form, sampled with NUTS',
       f'{BEN["divergences"]} divergences, R̂ ≤ {BEN["rhat_top200_max"]:.3f}, median ESS {BEN["ess_top200_median"]:.0f}.\n'
       f'{BEN["pct_credible"]:.1%} of weights credibly non-zero; each carries a 94% interval.\nAgreement with the frequentist fit: Jaccard {BEN["jaccard"]:.2f}.'),
      ('Stage 4: rank every OSM tag', 'score = (1 − λ) · noun similarity + λ · pattern table',
       'All 227 lexicon tags are scored for every description.\nλ is chosen by cross-validation (earlier work)\nor on a development set (final run).'),
      ('Stage 5: evaluate as the base paper does', 'Hits@k and nDCG@20, worst-case tie rule, reproduction before any claim',
       f'Base paper reproduced before anything new: W2V-clustered Hits@3 {r106b:.3f}\nagainst their published results (mean |Δ| 0.020).\nHits@3 is the headline: the place is found within three OSM queries.')]
for i, (t, sub, body) in enumerate(ST):
    s = slide(t, sub); T(s, body, 0.75, 2.4, 11.5, 2.6, 22); strip(s, i)

# 9 earlier results
s = slide('Earlier work: the gain on the base paper\'s own set and on a new Pakistani set', 'Hits@3; base = the base paper\'s method as reproduced here')
bars(s, [('base paper\n(106 set)', r106b, False), ('ours\n(106 set)', r106o, True), ('base paper\n(Pakistan n≥3)', pkb, False), ('ours\n(Pakistan n≥3)', pko, True)], 1.2, 2.3, 10.5, 3.3, 0.75)

# 10 why a final run
s = slide('Why one more run: the earlier evidence rested on hope in four places')
for i, (a, b) in enumerate([('λ tuned on the test sets', 'cross-validation softens it, but the test set still chose it'),
                            ('tiny test sets', '106 and 157 sentences: good-vs-good gaps are 3–10 sentences'),
                            ('weak training labels', 'labels guessed from type nouns inside the sentence'),
                            ('selection unproven', 'an unselected table matched the elastic net on both sets')]):
    box(s, 0.75 + i * 3.05, 2.2, 2.8, 1.0, BK, BK, a, 16, WH, True); T(s, b, 0.75 + i * 3.05, 3.35, 2.8, 1.6, 15, color=GR, align=PP_ALIGN.CENTER)

# 11 final protocol flow
s = slide('The final run: protocol fixed before any data was seen', f'OpenStreetMap description tags, labelled by the element\'s real OSM tag, type words masked')
flow = [(f'harvest\n{FUN[0]["rows"]:,}\ndescriptions', LG), (f'English, masked,\ndeduplicated\n{SPL["rows"]:,} rows', LG),
        (f'train\n{SPL["split_counts"]["train"]:,}', WH), (f'dev\n{SPL["split_counts"]["dev"]:,}\nchooses everything', WH), (f'test\n{SPL["split_counts"]["test"]:,}\nscored once', BK)]
for i, (t, fill) in enumerate(flow):
    box(s, 0.75 + i * 2.5, 2.6, 2.1, 1.6, fill, BK, t, 15, WH if fill == BK else BK, fill == BK)
    if i < 4: arrow(s, 0.75 + i * 2.5 + 2.13, 3.25, 0.34)
T(s, 'Split by OSM element and by place name · test ids hashed at split time · a change is kept only if its dev 95% CI excludes zero · comparisons Holm-corrected',
  0.75, 4.8, 11.8, 1.0, 15, color=GR)

# 12 previous vs final
s = slide('What changed between the earlier work and the final run')
rows_ = [('', 'earlier work', 'final OSM run'), ('training data', 'Wikivoyage, ~400k sentences', f'OSM descriptions, {SPL["split_counts"]["train"]:,}'),
         ('labels', 'weak: guessed from type nouns', 'strong: the element\'s real OSM tag'),
         ('test size', '106 and 157 sentences', f'{SPL["split_counts"]["test"]:,} held-out rows + both earlier sets'),
         ('λ and backbone chosen on', 'the test sets (cross-validation)', f'a separate dev set ({SPL["split_counts"]["dev"]:,})'),
         ('decision rule', 'best number', 'kept only if the dev 95% CI excludes zero'),
         ('overfitting check', 'none', 'learning curves over 10–100% of train')]
for j, row in enumerate(rows_):
    for k, cell in enumerate(row):
        T(s, cell, 0.75 + [0, 3.4, 7.6][k], 1.8 + j * 0.72, [3.3, 4.1, 5.0][k], 0.7, 16, bold=(j == 0 or k == 0), color=BK if (j == 0 or k != 1) else GR)

# 13 headline number
s = slide(f'On {FIN["sets"]["osm_test"]:,} held-out descriptions the pipeline doubles the base paper', 'Hits@3; the test set was scored once, after every choice was fixed on dev')
T(s, f'{F["osm_test"][K_BASE]["Hits@3"]:.3f}', 0.9, 2.3, 4.5, 1.8, 96, color=GR); T(s, 'base paper method', 0.9, 4.2, 4.5, 0.5, 18, color=GR)
T(s, f'{F["osm_test"][K_FIN]["Hits@3"]:.3f}', 6.4, 2.3, 5, 1.8, 96, True); T(s, 'this thesis', 6.4, 4.2, 5, 0.5, 18, True)
T(s, f'Δ {fb["delta_hits3"]:+.3f}   95% CI [{fb["ci95"][0]:+.3f}, {fb["ci95"][1]:+.3f}]   Holm p < 0.0001', 0.9, 5.2, 11.5, 0.6, 20)

# 14 figure: all configurations
s = slide('Every configuration on the held-out test', 'The base paper\'s own Algorithm 3 scores lower still on this set (post-hoc)')
s.shapes.add_picture(f'{FIG}/fig_4.20_heldout_test_hits3.png', Inches(1.4), Inches(2.0), height=Inches(5.0))

# 15 improvement over the base paper's claim, all sets
s = slide('The adjective claim holds on every evaluation set', 'Hits@3, base-paper method vs this thesis; same configuration everywhere, nothing tuned on these sets')
sets_ = [('OSM test', 'osm_test'), ('base-paper set', 'eval_106'), ('Pakistan n≥3', 'pk_n3')]
items = []
for lab, k in sets_: items += [(f'base\n{lab}', F[k][K_BASE]['Hits@3'], False), (f'ours\n{lab}', F[k][K_FIN]['Hits@3'], True)]
bars(s, items, 0.9, 2.2, 11.4, 3.2, 0.75)
p106, ppk = cmp_[('eval_106', K_FIN, K_BASE)]['p_holm'], cmp_[('pk_n3', K_FIN, K_BASE)]['p_holm']
T(s, f'Significant on the OSM test. Same direction on the two small sets, not significant after correction (Holm p {p106:.2f}, {ppk:.2f}).', 0.9, 6.55, 11.5, 0.6, 14, color=GR)

# 16 what each stage contributes
gd = {x['gate']: x for x in ADP['gates']}
s = slide('What each stage contributes, measured on dev', 'Paired-bootstrap Δ Hits@3 with 95% CI')
lines_ = [('mined adjective patterns vs nouns only', gd['tables at all: conf vs nouns only'], 'the gain'),
          ('Numberbatch vs Word2Vec', gd['backbone: numberbatch vs w2v (conf)'], 'kept'),
          ('elastic-net selection vs unselected table', gd['selection: freq EN C0.5 vs conf (numberbatch)'], 'no accuracy gain'),
          ('Bayesian EN vs unselected table', gd['selection: Bayesian EN vs conf (numberbatch)'], 'kept for intervals')]
for j, (lab, g_, verdict) in enumerate(lines_):
    y = 2.2 + j * 1.0; T(s, lab, 0.75, y, 5.4, 0.6, 18)
    T(s, f'{g_["delta_hits3"]:+.3f}  [{g_["ci95"][0]:+.3f}, {g_["ci95"][1]:+.3f}]', 6.3, y, 3.4, 0.6, 18, True)
    T(s, verdict, 9.9, y, 3, 0.6, 16, color=GR)
T(s, 'The Bayesian elastic net earns its place by giving every weight a credible interval: the system explains each prediction itself, without post-hoc XAI (RQ3).', 0.75, 6.35, 11.8, 0.8, 15, color=GR)

# 17 robustness + fit
C_, D_ = PH['C_near_duplicates'], PH['D_cue_leak']
s = slide('It holds up under scrutiny', 'Learning curves and post-hoc audits')
s.shapes.add_picture(f'{FIG}/fig_4.22_learning_curves.png', Inches(0.6), Inches(2.0), width=Inches(7.2))
T(s, (f'Train–dev gap closes to {LCJ["conf"][-1]["gap"]:.3f}: not overfitting.\n\n'
      f'Without the {C_["test_rows_with_train_near_dup"]} near-duplicate test rows:\n{C_["test_hits3_excluding_near_dups"]["final"]:.3f} vs {C_["test_hits3_excluding_near_dups"]["base_alg2"]:.3f}\n\n'
      f'Without the {D_["test_rows_leaking"]} rows that still hold a synonym:\n{D_["test_hits3_non_leaking"]["final"]:.3f} vs {D_["test_hits3_non_leaking"]["base_alg2"]:.3f}'), 8.2, 2.2, 4.6, 4.5, 16)

# 18 close
s = slide('What the thesis shows')
T(s, ('Adjectives, mined as patterns, are what the base paper was missing: they double its Hits@3 on held-out data.\n\n'
      'The pattern stage carries accuracy; the Bayesian elastic net carries explanation, each weight with its interval.\n\n'
      'Every choice was fixed on a development set; the test was scored once.\n\n'
      'Limits: small target-register sets, few Pakistani OSM rows, English only.'), 0.75, 1.8, 11.8, 5, 22)
out = f'{OUT}/slides/defence_deck.pptx'; prs.save(out); print('wrote', out, len(prs.slides), 'slides')

In [ ]:
import json, os
ROOT = '/content/drive/MyDrive/geoproject'; OUT = f'{ROOT}/osmfinal'
print(sorted(os.listdir(f'{ROOT}/results_h')))
for p in ['results_i/i2_fpgrowth_comparison.json'] + [f'results_h/{f}' for f in os.listdir(f'{ROOT}/results_h') if f.endswith('.json') and 'repro' not in f]:
    d = json.load(open(f'{ROOT}/{p}')); print('==', p)
    for k, v in (d.items() if isinstance(d, dict) else []): print(f'  {k}: {str(v)[:120]}')

In [ ]:
H3 = json.load(open(f'{ROOT}/results_h/h3_reproduction.json')); I2 = json.load(open(f'{ROOT}/results_i/i2_fpgrowth_comparison.json'))
ev = json.load(open(f'{ROOT}/eval/evalset_106_extracted.json')); ev = ev['rows'] if isinstance(ev, dict) else ev
pub, ours = H3['published']['W2V Clustered'], H3['W2V_clustered']
mad = sum(abs(ours[k] - pub[k]) for k in pub) / len(pub)
speed = I2['extended_40k']['time_s'] / I2['guided_40k']['time_s']
adj_share = sum(bool(r['adjectives']) for r in ev) / len(ev)
print('keys:', list(ev[0])); print(f'mean|Δ| {mad:.3f} | speedup {speed:.0f}x | rows with adjective {adj_share:.1%}')
for name, key in (('exact', lambda w: w.lower()), ('stem5', lambda w: w.lower()[:5])):
    hit = tot = 0
    for r in ev:
        mine = {key(w) for w in r.get('ours_nouns', [])}; theirs = [key(w) for w in r['context_nouns']]
        hit += sum(t in mine for t in theirs); tot += len(theirs)
    print(f'noun recall {name}: {hit/tot:.1%}')

In [ ]:
from nltk.stem import PorterStemmer
ps = PorterStemmer(); hit = tot = 0
for r in ev:
    mine = {ps.stem(w.lower()) for w in r.get('ours_nouns', [])}; theirs = [ps.stem(w.lower()) for w in r['context_nouns']]
    hit += sum(t in mine for t in theirs); tot += len(theirs)
REC = hit / tot; print(f'noun recall porter: {REC:.1%}')

In [ ]:
# R1c -- English only, mask the type words, drop thin rows, dedup, cap, group-split, hash the test set
import re, hashlib
import py3langid.langid as L
lid = L.LanguageIdentifier.from_model_file(L.MODEL_FILE, norm_probs=True)
d = el.copy()
d['description'] = d['description'].str.replace(r'\s+', ' ', regex=True).str.strip()
pred = [lid.classify(t) for t in d['description'].tolist()]
d['lang'] = [p[0] for p in pred]; d['lang_p'] = [float(p[1]) for p in pred]
en = d[(d['lang'] == 'en') & (d['lang_p'] >= 0.6)].copy()
print(f'English (p>=.6): {len(en):,} of {len(d):,} | Pakistan {en.pakistan.sum():,}')

def mask_pattern(tag, value):
    phrases = {tag.replace('-', ' '), str(value).replace('_', ' ')}
    alts = sorted({re.escape(p).replace(r'\ ', r'[\s_-]?') for p in phrases if p}, key=len, reverse=True)
    return re.compile(r'\b(?:' + '|'.join(alts) + r')(?:e?s)?\b', re.I)
PAT = {}
def mask(row):
    k = (row.tag, row.value)
    if k not in PAT: PAT[k] = mask_pattern(*k)
    return PAT[k].sub(' ', row.description)
en['text'] = [mask(r) for r in en.itertuples(index=False)]
en['text'] = en['text'].str.replace(r'\s+', ' ', regex=True).str.strip()
en['n_words'] = en['text'].str.count(r'\b[A-Za-z]{2,}\b')
en = en[en['n_words'] >= 4]
print(f'>=4 content words after masking: {len(en):,}')

en['dedup_key'] = en['text'].str.lower().str.replace(r'\d+', '#', regex=True).str.replace(r'[^a-z# ]', '', regex=True)
en = en.drop_duplicates('dedup_key')
print(f'after exact/template dedup: {len(en):,} | Pakistan {en.pakistan.sum():,}')

CAP, MIN_PER_TAG = 3000, 10
en = en.sample(frac=1.0, random_state=0)
capped = pd.concat([pd.concat([g[g.pakistan], g[~g.pakistan].head(max(0, CAP - int(g.pakistan.sum())))])
                    for _, g in en.groupby('tag')])
vc = capped['tag'].value_counts(); capped = capped[capped['tag'].isin(vc[vc >= MIN_PER_TAG].index)]
print(f'capped at {CAP}/tag, tags with >= {MIN_PER_TAG} rows: {len(capped):,} rows, {capped.tag.nunique()} tags')

# group = normalised name when present, else the element: the same named place never straddles splits
grp = capped['name'].fillna('').str.lower().str.strip()
grp = grp.where(grp != '', capped['osm_id'])
h = grp.map(lambda s: int(hashlib.sha256(s.encode()).hexdigest()[:8], 16) % 100)
capped['split'] = pd.cut(h, [-1, 79, 89, 99], labels=['train', 'dev', 'test']).astype(str)
for a, b in (('train', 'dev'), ('train', 'test'), ('dev', 'test')):
    assert not (set(grp[capped.split == a]) & set(grp[capped.split == b])), f'group overlap {a}/{b}'
test_hash = hashlib.sha256('\n'.join(sorted(capped.loc[capped.split == 'test', 'osm_id'])).encode()).hexdigest()
cols = ['osm_id', 'tag', 'key', 'value', 'text', 'n_words', 'pakistan', 'split']
capped[cols].to_parquet(f'{OUT}/osm_corpus.parquet')
stats = {'rows': len(capped), 'tags': int(capped.tag.nunique()), 'cap_per_tag': CAP, 'min_per_tag': MIN_PER_TAG,
         'split_counts': capped.split.value_counts().to_dict(),
         'pakistan_by_split': capped[capped.pakistan].split.value_counts().to_dict(),
         'test_sha256_of_sorted_ids': test_hash, 'lang_filter': 'py3langid 0.4.0 en, normalised p>=0.6',
         'masking': 'tag phrase and OSM value phrase, optional plural', 'min_content_words': 4}
json.dump(stats, open(f'{OUT}/split_stats.json', 'w'), indent=1)
print(json.dumps(stats, indent=1))